## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** el concepto de *transfer learning* y cuándo conviene aplicarlo.
2. **Reutilizar** un modelo preentrenado como extractor de características en TensorFlow/Keras.
3. **Realizar** *fine-tuning* de un modelo para una tarea específica.
4. **Comparar** entrenar desde cero frente a transferir conocimiento.

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb).

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf

In [ ]:
dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)

In [ ]:
test_set, valid_set, train_set = tfds.load("tf_flowers",split=["train[0%:10%]", "train[10%:25%]", "train[25%:]"], as_supervised=True )

In [ ]:
# import tensorflow as tf

In [ ]:
# Previo a la utilización de nuestro dataset en la red neuronal, la red que utilizaremos provee de
# funcionalidades de pre-procesamiento ya incluídas. Por lo tanto, utilizaremos estas funcionalidades
# y las dejaremos dentro de una función
def preprocess(image, label):
    # Cambiaremos las dimensiones de la imagen de entrada
    resized_image = tf.image.resize(image, [224, 224]) # Guardamos la imagen con nuevas dimensiones 224x224
    # Luego pasamos la imagen modificada en tamaño al preprocesamiento de nuestra red. La red
    # que utilizaremos de ejemplo tiene por nombre Xception
    final_image = tf.keras.applications.xception.preprocess_input(resized_image)
    # Finalmente se retorna una imagen pre procesada (según lo indique el preprocess de xception)
    # y su etiqueta
    return final_image, label

In [ ]:
# En este apartado hacemos los batch de datos directamente desde el dataset cargado por tensorflow
batch_size = 32

# Mezclamos el dataset 
train_set = train_set.shuffle(1000)
# Tanto para training, test y validación, aplicamos la función de preprocesamiento (preprocess)
# y luego generamos los batch de datos
train_set = train_set.map(preprocess).batch(batch_size).prefetch(1)
test_set = test_set.map(preprocess).batch(batch_size).prefetch(1)
valid_set = valid_set.map(preprocess).batch(batch_size).prefetch(1)

In [ ]:
# Acá empezamos con transfer learning. Lo que haremos será cargar una arquitectura de red neuronal desde
# tensorflow ya entrenada. Eso quiere decir que, no solo estamos cargando la arquitectura, con sus neuronas
# y conexiones, sino que también estamos cargando los PESOS DE ENTRENAMIENTO.

# Weights indica si utilizaremos pesos pre entrenados con el dataset imagenet o no
# include_top es el parámetro que indica explícitamente si quieres o no la capa de salida original
# de esta red.
# En base_model tenemos cargado nuestro modelo Xception sin la capa de salida. Nosotros podemos
# poner NUESTRA PROPIA CAPA(S) DE SALIDA
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)
# Acá agregamos nuestras capas adicionales
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(5, activation="softmax")(avg)

# Podemos conectar el input de nuestro modelo base con el output recién generado a través de 
# un modelo de Keras
model = tf.keras.Model(inputs=base_model.input, outputs=output)
# En este punto tenemos un nuevo modelo que utiliza como base la arquitectura Xception

In [ ]:
# También nosotros conversamos que es posible decidir si queremos reentrenar aquellas capas ya entrenadas

# Acá vamos capa por capa modificando el parámetro "trainable" que permite (o no) reentrenar dicha capa
for layer in base_model.layers:
    layer.trainable = False # Esto impide que las capas se re entrenen

# Con este for, sobre todas las capas de nuestro modelo, estamos impidiendo que se reentrene
# alguna de sus capas ya entrenadas

In [ ]:
# Proceso de compilación  (tal cual vimos en las clases anteriores)
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", 
              metrics=["accuracy"])

In [ ]:
history = model.fit(train_set, epochs=10, validation_data=valid_set)

In [ ]:
model.predict(test_set)

In [ ]:
import numpy as np
for image, label in test_set:
    print("Etiqueta real",label)
    print("Predicción", model.predict(image))
